# 🚀 Cloud Training Notebook: AP-2026-Q-2 (0% Local RAM)
Este notebook permite entrenar modelos pesados (CatBoost Depth 8/10 en GPU, Redes Neuronales Multimodales PyTorch, Stacking) en la nube usando **Google Colab (GPU T4 / CPU 16GB)** de forma 100% gratuita y sin consumir un solo megabyte de RAM en la máquina local.

### Pipeline Integrado:
1. Clona el branch de coordinación `antigravity/overnight-recovery-coordination-20261030`.
2. Conecta con Kaggle API para descargar los datasets si no están presentes.
3. Entrena modelos de alta capacidad con 5-Fold Stratified CV (sin seller_id leak).
4. Genera predicciones OOF y de Test, y las sube automáticamente al repositorio.

In [ ]:
# Paso 1: Clonar repositorio
!git clone -b antigravity/overnight-recovery-coordination-20261030 https://github.com/arirakowitba/tp_ap.git
%cd tp_ap
!pip install lightgbm xgboost catboost scikit-learn pandas numpy kaggle psutil

In [ ]:
# Paso 2: Autenticación de Kaggle (Opcional si subes los jsonlines directo a data/ap_2026_q_2/raw)
import os, json
from google.colab import files

raw_dir = 'data/ap_2026_q_2/raw'
os.makedirs(raw_dir, exist_ok=True)

if not os.path.exists(os.path.join(raw_dir, 'train_data.jsonlines')):
    print('Sube tu kaggle.json o sube directamente train_data.jsonlines a la carpeta data/ap_2026_q_2/raw:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn == 'kaggle.json':
            !mkdir -p ~/.kaggle
            !mv kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle competitions download -c ap-2026-q-2 -p data/ap_2026_q_2
            !unzip -o data/ap_2026_q_2/*.zip -d data/ap_2026_q_2/raw/
            break
print('Verificación de datos:')
!ls -lh data/ap_2026_q_2/raw/

In [ ]:
# Paso 3: Entrenar CatBoost GPU Depth 8 (Zero Local RAM)
import json, time, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

print('Cargando datasets en memoria Colab...')
with open('data/ap_2026_q_2/raw/train_data.jsonlines', encoding='utf-8') as f:
    tr_data = [json.loads(line) for line in f]
with open('data/ap_2026_q_2/raw/test_data.jsonlines', encoding='utf-8') as f:
    te_data = [json.loads(line) for line in f]

tr_titles = [d.get('title', '') for d in tr_data]
te_titles = [d.get('title', '') for d in te_data]
y_train = np.array([1 if d['condition'] == 'new' else 0 for d in tr_data], dtype=np.int8)

print('Vectorización TF-IDF + SVD (150 componentes)...')
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=120000, sublinear_tf=True)
X_all = vec.fit_transform(tr_titles + te_titles)
svd = TruncatedSVD(n_components=150, random_state=42)
X_svd = svd.fit_transform(X_all)
X_tr = X_svd[:len(tr_titles)]
X_te = X_svd[len(tr_titles):]

oof_cat = np.zeros(len(tr_titles), dtype=np.float32)
test_cat = np.zeros(len(te_titles), dtype=np.float32)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=777)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_tr, y_train)):
    print(f'=== Entrenando Fold {fold+1}/5 ===')
    clf = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.03,
        depth=8,
        task_type='GPU',
        random_seed=777+fold,
        verbose=500
    )
    clf.fit(X_tr[train_idx], y_train[train_idx], eval_set=(X_tr[val_idx], y_train[val_idx]), early_stopping_rounds=150)
    v_p = clf.predict_proba(X_tr[val_idx])[:, 1]
    oof_cat[val_idx] = v_p
    test_cat += clf.predict_proba(X_te)[:, 1] / 5.0
    print(f'Fold {fold+1} Acc: {accuracy_score(y_train[val_idx], (v_p >= 0.5).astype(int)):.5f}')

overall_acc = accuracy_score(y_train, (oof_cat >= 0.5).astype(int))
overall_auc = roc_auc_score(y_train, oof_cat)
print(f'\n>>> RESULTADO FINAL CATBOOST GPU: OOF Acc = {overall_acc:.5f} | AUC = {overall_auc:.5f} <<<')

os.makedirs('results/cloud_catboost_gpu_depth8', exist_ok=True)
np.savez_compressed('results/cloud_catboost_gpu_depth8/catboost_predictions.npz', oof_cat=oof_cat, test_cat=test_cat)
pd.DataFrame({'ID': range(1, 30001), 'prob_new': test_cat}).to_csv('results/cloud_catboost_gpu_depth8/test_probabilities.csv', index=False)
print('Predicciones guardadas exitosamente.')